In [1]:
# !pip install -U langchain langchain-openai langchain-chroma langchain-cohere langchain-text-splitters pypdf python-dotenv

In [5]:
%pip install -U langchain-chroma langchain-cohere

  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached protobuf-7.36.2-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 2.1 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 1.9 MB/s  0:00:06 eta 0:00:01
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 1.5 MB/s  0:00:05 eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 2.0 MB/s  0:00:10m0:00:0100:01
Using cached protobuf-7.36.2-cp310-abi3-macosx_10_9_universal2.whl (456 kB)
Using cached referencing-0.37.0-py3-none-any.whl (26 kB)
 

In [2]:
import re
from pathlib import Path
from collections import Counter
from dotenv import load_dotenv
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""],
)

# --- Podcast: one long text, cut into chunks ---
podcast_text = Path("podcast.txt").read_text()
podcast_docs = splitter.create_documents(
    [podcast_text],
    metadatas=[{"source": "podcast", "section": "trustworthy_ai_guidelines"}],
)

# --- EU AI Act: recitals / articles / annexes, articles split at their headings ---
reader = PdfReader("eu_ai_act.pdf")
ARTICLE = re.compile(r"^Ar\s?ticle\s+(\d+)\s*$")
act_docs = []
current = None

for page_no, page in enumerate(reader.pages, start=1):
    if page_no <= 43:
        section = "recitals"
    elif page_no <= 123:
        section = "articles"
    else:
        section = "annexes"
    lines = (page.extract_text() or "").splitlines()

    if section != "articles":
        for chunk in splitter.split_text("\n".join(lines)):
            act_docs.append(Document(page_content=chunk, metadata={
                "source": "eu_ai_act", "section": section, "page": page_no, "article": 0}))
        continue

    for i, line in enumerate(lines):
        m = ARTICLE.match(line.strip())
        if m:
            if current:
                act_docs.append(current)
            title = lines[i + 1].strip() if i + 1 < len(lines) else ""
            current = Document(page_content="", metadata={
                "source": "eu_ai_act", "section": "articles", "page": page_no,
                "article": int(m.group(1)), "title": title})
        if current:
            current.page_content += line + "\n"

if current:
    act_docs.append(current)

article_docs = [d for d in act_docs if d.metadata["section"] == "articles"]
other_docs = [d for d in act_docs if d.metadata["section"] != "articles"]
all_docs = podcast_docs + other_docs + splitter.split_documents(article_docs)

print("Total chunks:", len(all_docs))
print(Counter(d.metadata["section"] for d in all_docs))
print(all_docs[-400].metadata)

Total chunks: 794
Counter({'articles': 407, 'recitals': 308, 'annexes': 60, 'trustworthy_ai_guidelines': 19})
{'source': 'eu_ai_act', 'section': 'articles', 'page': 45, 'article': 2, 'title': 'Scope'}


In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(all_docs, embeddings, collection_name="trustworthy_ai")
print("Stored:", vectorstore._collection.count(), "chunks")

Stored: 794 chunks


In [8]:
def show(results):
    for i, (doc, score) in enumerate(results, 1):
        m = doc.metadata
        where = f"Art. {m['article']} – {m.get('title', '')}" if m.get("article") else m["section"]
        print(f"{i}. {score:.3f} | {m['source']} | {where} | p.{m.get('page', '-')}")
        print("   ", doc.page_content[:150].replace("\n", " "), "...")

test_queries = [
    ("Which AI practices are prohibited?", "EU AI Act, Article 5"),
    ("What does human in the loop mean?", "Podcast (HITL / HOTL / HIC)"),
    ("Must a chatbot tell users that it is an AI?", "EU AI Act, Article 50"),
    ("When do the rules of the AI Act start to apply?", "EU AI Act, Article 113"),
]

In [9]:
for q, expected in test_queries:
    print(f"\n❓ {q}\n   expected: {expected}")
    show(vectorstore.similarity_search_with_relevance_scores(q, k=5))


❓ Which AI practices are prohibited?
   expected: EU AI Act, Article 5
1. 0.664 | eu_ai_act | Art. 5 – Prohibited AI practices | p.51
    Article 5 Prohibited AI practices 1. The following AI practices shall be prohibited: (a) the placing on the market, the putting into service or the us ...
2. 0.586 | eu_ai_act | Art. 4 – AI literacy | p.51
    Article 4 AI literacy Providers and deployers of AI systems shall take measures to ensure, to their best extent, a sufficient level of AI literacy of  ...
3. 0.475 | eu_ai_act | recitals | p.9
    intention to distort behaviour where the distortion results from factors external to the AI system which are outside  the control of the provider or t ...
4. 0.446 | eu_ai_act | recitals | p.8
    manipulative, exploitative and social control practices. Such practices are particularly harmful and abusive and  should be prohibited because they co ...
5. 0.421 | eu_ai_act | recitals | p.12
    generalisability. Therefore, AI systems identifying or infe

In [10]:
from langchain_cohere import CohereRerank

reranker = CohereRerank(model="rerank-v3.5", top_n=5)

def rerank_search(query, k_initial=20, top_n=5):
    candidates = vectorstore.similarity_search(query, k=k_initial)
    reranked = reranker.compress_documents(candidates, query)
    return [(d, d.metadata["relevance_score"]) for d in reranked[:top_n]]

for q, expected in test_queries:
    print(f"\n❓ {q}\n   expected: {expected}")
    show(rerank_search(q))


❓ Which AI practices are prohibited?
   expected: EU AI Act, Article 5
1. 0.948 | eu_ai_act | Art. 5 – Prohibited AI practices | p.51
    Article 5 Prohibited AI practices 1. The following AI practices shall be prohibited: (a) the placing on the market, the putting into service or the us ...
2. 0.853 | eu_ai_act | recitals | p.9
    intention to distort behaviour where the distortion results from factors external to the AI system which are outside  the control of the provider or t ...
3. 0.794 | eu_ai_act | recitals | p.12
    generalisability. Therefore, AI systems identifying or inferring emotions or intentions of natural persons on the basis  of their biometric data may l ...
4. 0.761 | eu_ai_act | recitals | p.9
    irrespective of whether they are put in place through AI systems or otherwise. The prohibitions of manipulative and  exploitative practices in this Re ...
5. 0.760 | eu_ai_act | recitals | p.8
    manipulative, exploitative and social control practices. Such practices 

In [11]:
def label(doc):
    m = doc.metadata
    return f"Art.{m['article']}" if m.get("article") else f"{m['source']}/{m['section']} p.{m.get('page', '-')}"

for q, expected in test_queries:
    base = vectorstore.similarity_search_with_relevance_scores(q, k=5)
    rr = rerank_search(q)
    print(f"\n❓ {q}   (expected: {expected})")
    print(f"   {'BASELINE':38} | RERANKED")
    for (bd, bs), (rd, rs) in zip(base, rr):
        print(f"   {bs:.3f} {label(bd):32} | {rs:.3f} {label(rd)}")


❓ Which AI practices are prohibited?   (expected: EU AI Act, Article 5)
   BASELINE                               | RERANKED
   0.664 Art.5                            | 0.948 Art.5
   0.586 Art.4                            | 0.853 eu_ai_act/recitals p.9
   0.475 eu_ai_act/recitals p.9           | 0.794 eu_ai_act/recitals p.12
   0.446 eu_ai_act/recitals p.8           | 0.761 eu_ai_act/recitals p.9
   0.421 eu_ai_act/recitals p.12          | 0.760 eu_ai_act/recitals p.8

❓ What does human in the loop mean?   (expected: Podcast (HITL / HOTL / HIC))
   BASELINE                               | RERANKED
   0.287 podcast/trustworthy_ai_guidelines p.- | 0.776 podcast/trustworthy_ai_guidelines p.-
   0.168 podcast/trustworthy_ai_guidelines p.- | 0.725 podcast/trustworthy_ai_guidelines p.-
   0.145 eu_ai_act/recitals p.21          | 0.454 podcast/trustworthy_ai_guidelines p.-
   0.134 podcast/trustworthy_ai_guidelines p.- | 0.094 podcast/trustworthy_ai_guidelines p.-
   0.122 podcast/trustwort

In [12]:
cands = vectorstore.similarity_search(test_queries[3][0], k=20)
print([label(d) for d in cands])
print("Art. 113 found:", any(d.metadata.get("article") == 113 for d in cands))

['Art.5', 'Art.106', 'Art.50', 'Art.111', 'Art.103', 'Art.56', 'Art.56', 'Art.105', 'Art.2', 'Art.95', 'Art.108', 'Art.111', 'eu_ai_act/recitals p.7', 'eu_ai_act/recitals p.3', 'eu_ai_act/recitals p.7', 'eu_ai_act/recitals p.7', 'eu_ai_act/recitals p.6', 'Art.94', 'Art.110', 'Art.2']
Art. 113 found: False


In [13]:
def filtered_rerank(query, where, k_initial=20, top_n=5):
    candidates = vectorstore.similarity_search(query, k=k_initial, filter=where)
    reranked = reranker.compress_documents(candidates, query)
    return [(d, d.metadata["relevance_score"]) for d in reranked[:top_n]]

print("Q3, only the EU AI Act:")
show(filtered_rerank(test_queries[2][0], {"source": "eu_ai_act"}))

print("\nQ4, only articles:")
q4 = test_queries[3][0]
cands = vectorstore.similarity_search(q4, k=20, filter={"section": "articles"})
print("Art. 113 in candidates:", any(d.metadata.get("article") == 113 for d in cands))
show(filtered_rerank(q4, {"section": "articles"}))

Q3, only the EU AI Act:
1. 0.238 | eu_ai_act | recitals | p.8
    sufficiently important adverse impacts on physical, psychological health or financial interests are likely to occur, are  particularly dangerous and s ...
2. 0.208 | eu_ai_act | recitals | p.33
    database. In order to maximise the availability and use of the EU database by the public, the EU database, including  the information made available t ...
3. 0.204 | eu_ai_act | recitals | p.31
    (119) Considering the quick pace of innovation and the technological evolution of digital services in scope of different  instruments of Union law in  ...
4. 0.153 | eu_ai_act | recitals | p.34
    and distinguishably disclose that the content has been artificially created or manipulated by labelling the AI output  accordingly and disclosing its  ...
5. 0.116 | eu_ai_act | Art. 50 – Transparency obligations for providers and deployers of certain AI systems | p.82
    disclose that the content has been artificially generated or manip

In [14]:
print("\nQ4 reworded:")
show(filtered_rerank("When does the AI Act enter into force and apply?", {"section": "articles"}))


Q4 reworded:
1. 0.462 | eu_ai_act | Art. 105 – Amendment to Directive 2014/90/EU | p.119
    2020/1828 (Artificial Intelligence Act) (OJ L, 2024/1689, 12.7.2024, ELI: http://data.europa.eu/eli/reg/  2024/1689/oj).’. ...
2. 0.420 | eu_ai_act | Art. 106 – Amendment to Directive (EU) 2016/797 | p.119
    2020/1828 (Artificial Intelligence Act) (OJ L, 2024/1689, 12.7.2024, ELI: http://data.europa.eu/eli/reg/  2024/1689/oj).’. OJ L, 12.7.2024 EN ELI: htt ...
3. 0.410 | eu_ai_act | Art. 103 – Amendment to Regulation (EU) No 167/2013 | p.118
    2020/1828 (Artificial Intelligence Act) (OJ L, 2024/1689, 12.7.2024, ELI: http://data.europa.eu/eli/reg/  2024/1689/oj).’. EN OJ L, 12.7.2024 118/144  ...
4. 0.349 | eu_ai_act | Art. 110 – Amendment to Directive (EU) 2020/1828 | p.121
    Article 110 Amendment to Directive (EU) 2020/1828 In Annex I to Directive (EU) 2020/1828 of the European Parliament and of the Council (58), the follo ...
5. 0.337 | eu_ai_act | Art. 111 – AI systems already placed 

In [15]:
# 1. Are Article 113's chunks in the database, and what do they say?
got = vectorstore.get(where={"article": 113})
print("Art. 113 chunks:", len(got["documents"]))
for t in got["documents"]:
    print(t[:400].replace("\n", " "), "\n---")

# 2. At which rank does Art. 113 actually appear?
res = vectorstore.similarity_search_with_relevance_scores(
    test_queries[3][0], k=200, filter={"section": "articles"})
for rank, (d, s) in enumerate(res, 1):
    if d.metadata.get("article") == 113:
        print(f"Art. 113 at rank {rank}, score {s:.3f}")

Art. 113 chunks: 1
Article 113 Entry into force and application This Regulation shall enter into force on the twentieth day following that of its publication in the Official Journal of the  European Union. It shall apply from 2 August 2026. However: (a) Chapters I and II shall apply from 2 February 2025; (b) Chapter III Section 4, Chapter V, Chapter VII and Chapter XII and Article 78 shall apply from 2 August 2025,  
---


In [16]:
import numpy as np

def cos(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

q = embeddings.embed_query(test_queries[3][0])
top_doc = res[0][0]

print("Art. 113 score:", round(cos(q, embeddings.embed_query(got["documents"][0])), 3))
print(f"Top hit (Art. {top_doc.metadata.get('article')}) score:",
      round(cos(q, embeddings.embed_query(top_doc.page_content)), 3))
print("Total results checked:", len(res))

Art. 113 score: 0.392
Top hit (Art. 5) score: 0.573
Total results checked: 200


In [17]:
FOOTER = re.compile(r"^(OJ L, \d|ELI: http|EN$|\d+/144$)")
SIGNATURE = re.compile(r"Done at Br\s?ussels")

def clean(text):
    text = SIGNATURE.split(text)[0]
    return "\n".join(line for line in text.splitlines() if not FOOTER.match(line.strip()))

clean_docs = []
for d in all_docs:
    m = d.metadata
    text = clean(d.page_content)
    if m.get("article") and not text.startswith("Article"):
        text = f"Article {m['article']} – {m['title']}\n{text}"
    clean_docs.append(Document(page_content=text, metadata=m))

vectorstore_v1 = vectorstore
vectorstore = Chroma.from_documents(clean_docs, embeddings, collection_name="trustworthy_ai_v2")
print("v2 stored:", vectorstore._collection.count(), "chunks")

v2 stored: 794 chunks


In [18]:
def find_rank(store, query, target, where=None, k=200):
    res = store.similarity_search_with_relevance_scores(query, k=k, filter=where)
    for rank, (d, s) in enumerate(res, 1):
        if d.metadata.get("article") == target:
            return f"rank {rank} (score {s:.3f})"
    return f"not in top {k}"

for name, store in [("v1 (raw)", vectorstore_v1), ("v2 (clean)", vectorstore)]:
    print(name)
    print("   Q4 → Art. 113:", find_rank(store, test_queries[3][0], 113, {"section": "articles"}))
    print("   Q3 → Art. 50: ", find_rank(store, test_queries[2][0], 50, {"source": "eu_ai_act"}))

v1 (raw)
   Q4 → Art. 113: not in top 200
   Q3 → Art. 50:  rank 2 (score 0.286)
v2 (clean)
   Q4 → Art. 113: not in top 200
   Q3 → Art. 50:  rank 8 (score 0.240)


In [19]:
print("Q3 (v2, EU AI Act only, reranked):")
show(filtered_rerank(test_queries[2][0], {"source": "eu_ai_act"}))

print("\nQ4 (v2, articles only, reranked):")
show(filtered_rerank(test_queries[3][0], {"section": "articles"}))

Q3 (v2, EU AI Act only, reranked):
1. 0.238 | eu_ai_act | recitals | p.8
    sufficiently important adverse impacts on physical, psychological health or financial interests are likely to occur, are  particularly dangerous and s ...
2. 0.208 | eu_ai_act | recitals | p.33
    database. In order to maximise the availability and use of the EU database by the public, the EU database, including  the information made available t ...
3. 0.204 | eu_ai_act | recitals | p.31
    (119) Considering the quick pace of innovation and the technological evolution of digital services in scope of different  instruments of Union law in  ...
4. 0.153 | eu_ai_act | recitals | p.34
    and distinguishably disclose that the content has been artificially created or manipulated by labelling the AI output  accordingly and disclosing its  ...
5. 0.133 | eu_ai_act | recitals | p.33
    system, unless this is obvious from the point of view of a natural person who is reasonably well-informed, observant  and circumspe

In [20]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def rag_answer(query, use_rerank=True, where=None):
    if use_rerank:
        hits = filtered_rerank(query, where) if where else rerank_search(query)
    else:
        hits = vectorstore.similarity_search_with_relevance_scores(query, k=5, filter=where)
    context = "\n\n".join(f"[{i}] ({label(d)}) {d.page_content}" for i, (d, _) in enumerate(hits, 1))
    prompt = f"""Answer the question using ONLY the sources below. Cite sources like [1], [2].
If the sources do not contain the answer, say: "The retrieved sources do not answer this."

Sources:
{context}

Question: {query}"""
    return llm.invoke(prompt).content, hits

In [21]:
import textwrap
for q, expected in test_queries:
    print(f"\n{'=' * 90}\n❓ {q}   (expected: {expected})")
    for mode in [False, True]:
        answer, hits = rag_answer(q, use_rerank=mode)
        print(f"\n--- {'RERANKED' if mode else 'BASELINE'} | sources: {[label(d) for d, _ in hits]}")
        print(textwrap.fill(answer, 110))


❓ Which AI practices are prohibited?   (expected: EU AI Act, Article 5)

--- BASELINE | sources: ['Art.5', 'Art.4', 'Art.5', 'Art.5', 'Art.5']
The prohibited AI practices include:  1. The use of AI systems that deploy subliminal techniques or
manipulative techniques to distort the behavior of individuals, impairing their ability to make informed
decisions and causing significant harm [1]. 2. The exploitation of vulnerabilities of individuals or specific
groups due to age, disability, or social/economic situations, leading to harmful behavior distortion [1]. 3.
The use of AI systems for making risk assessments of individuals based solely on profiling or personality
traits, unless supporting human assessment based on objective facts linked to criminal activity [3]. 4. The
evaluation or classification of individuals based on social behavior or inferred/predicted characteristics
that leads to detrimental treatment in unrelated social contexts [4]. 5. The creation or expansion of facial
re